# Granite Peak CRE: 06 Lakebase operational serving

The Databricks App reads and writes **Lakebase** (managed Postgres), not the SQL warehouse:

| Postgres table | Source | Use in the app |
|---|---|---|
| `property_360` | snapshot-synced from gold `property_360` | Property page: one-row lookups in milliseconds |
| `lease_abstract` | snapshot-synced from gold `lease_abstract` | Review queue: extracted fields + confidence |
| `lease_discrepancies` | snapshot-synced from gold `lease_discrepancies` | Review queue: PDF vs. ERP mismatches |
| `renewal_scores` | snapshot-synced from `renewal_scores` | Renewal probability and vacancy loss per lease |
| `app.abstract_review` | native Postgres table | Analyst corrections written by the app |

The last cell copies approved corrections from Postgres into the Delta table `abstract_review`. The next pipeline update applies them in `lease_abstract`, so Genie and the dashboards see the fix.

In [1]:
%pip install -q "psycopg[binary]>=3.1" "databricks-sdk>=0.60"
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")
dbutils.widgets.text("instance", "granite-cre", "Lakebase instance")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
INSTANCE = dbutils.widgets.get("instance").strip()
uc = f"{catalog}.{schema}"
PG_DB = "databricks_postgres"
SYNCED = {"property_360": ["property_id"], "lease_abstract": ["lease_id"],
          "lease_discrepancies": ["lease_id"], "renewal_scores": ["lease_id"]}
print(f"Source: {uc}  |  Lakebase instance: {INSTANCE}")

Source: smmorgan_catalog.granite_commercial_real_estate  |  Lakebase instance: granite-cre


In [3]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance

w = WorkspaceClient()
try:
    inst = w.database.get_database_instance(INSTANCE)
    print(f"instance exists: {inst.name} ({inst.state})")
except Exception:
    w.database.create_database_instance(DatabaseInstance(name=INSTANCE, capacity="CU_1"))
    print(f"creating instance {INSTANCE} ...")
while (inst := w.database.get_database_instance(INSTANCE)).state.value != "AVAILABLE":
    time.sleep(20)
print(f"instance {inst.name}: {inst.state.value}, capacity {inst.capacity}, pg {inst.pg_version}")

instance exists: granite-cre (DatabaseInstanceState.AVAILABLE)
instance granite-cre: AVAILABLE, capacity CU_1, pg PG_VERSION_16


In [4]:
import uuid, psycopg

def pg_connect():
    cred = w.database.generate_database_credential(request_id=str(uuid.uuid4()), instance_names=[INSTANCE])
    return psycopg.connect(host=inst.read_write_dns, dbname=PG_DB, user=w.current_user.me().user_name,
                           password=cred.token, sslmode="require", autocommit=True)

from databricks.sdk.service.database import (NewPipelineSpec, SyncedDatabaseTable,
                                             SyncedTableSchedulingPolicy, SyncedTableSpec)

# Create missing synced tables; refresh existing snapshot tables by re-running their sync pipeline
sync_updates = {}
for table, pk in SYNCED.items():
    name = f"{uc}.{table}_synced"
    try:
        st = w.database.get_synced_database_table(name)
        pipeline_id = st.data_synchronization_status.pipeline_id
        sync_updates[table] = (pipeline_id, w.pipelines.start_update(pipeline_id, full_refresh=True).update_id)
        print(f"refresh started: {uc}.{table} -> {name}")
    except Exception as e:
        if "does not exist" not in str(e).lower() and "not found" not in str(e).lower():
            raise
        with pg_connect() as conn:  # a synced table deleted from Unity Catalog can leave its Postgres table behind
            conn.execute(f'DROP TABLE IF EXISTS "{schema}"."{table}_synced"')
        w.database.create_synced_database_table(SyncedDatabaseTable(
            name=name, database_instance_name=INSTANCE, logical_database_name=PG_DB,
            spec=SyncedTableSpec(source_table_full_name=f"{uc}.{table}", primary_key_columns=pk,
                                 scheduling_policy=SyncedTableSchedulingPolicy.SNAPSHOT,
                                 new_pipeline_spec=NewPipelineSpec(storage_catalog=catalog, storage_schema=schema))))
        print(f"sync created: {uc}.{table} -> {name}")

for table, (pipeline_id, update_id) in sync_updates.items():
    while (state := w.pipelines.get_update(pipeline_id, update_id).update.state.value) not in ("COMPLETED", "FAILED", "CANCELED"):
        time.sleep(15)
    print(f"{table}_synced refresh: {state}")
    assert state == "COMPLETED", f"{table} sync {state}"

sync created: smmorgan_catalog.granite_commercial_real_estate.property_360 -> smmorgan_catalog.granite_commercial_real_estate.property_360_synced
refresh started: smmorgan_catalog.granite_commercial_real_estate.lease_abstract -> smmorgan_catalog.granite_commercial_real_estate.lease_abstract_synced
refresh started: smmorgan_catalog.granite_commercial_real_estate.lease_discrepancies -> smmorgan_catalog.granite_commercial_real_estate.lease_discrepancies_synced
refresh started: smmorgan_catalog.granite_commercial_real_estate.renewal_scores -> smmorgan_catalog.granite_commercial_real_estate.renewal_scores_synced
lease_abstract_synced refresh: COMPLETED
lease_discrepancies_synced refresh: COMPLETED
renewal_scores_synced refresh: COMPLETED


In [5]:
# Poll Postgres for rows rather than trusting the provisioning state alone
deadline = time.time() + 1800
pending = set(SYNCED)
while pending and time.time() < deadline:
    with pg_connect() as conn:
        for table in sorted(pending):
            try:
                n = conn.execute(f'SELECT count(*) FROM "{schema}"."{table}_synced"').fetchone()[0]
                if n > 0:
                    print(f"{table}_synced: {n:,} rows in Postgres"); pending.discard(table)
            except psycopg.Error:
                pass
    if pending:
        time.sleep(30)
assert not pending, f"synced tables not ready: {pending}"
for table in SYNCED:
    st = w.database.get_synced_database_table(f"{uc}.{table}_synced").data_synchronization_status
    print(f"{table}_synced: {st.detailed_state.value}")

lease_abstract_synced: 116 rows in Postgres
lease_discrepancies_synced: 16 rows in Postgres
property_360_synced: 118 rows in Postgres
renewal_scores_synced: 391 rows in Postgres
property_360_synced: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
lease_abstract_synced: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
lease_discrepancies_synced: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
renewal_scores_synced: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE


In [6]:
with pg_connect() as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS app")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS app.abstract_review (
          lease_id        text        NOT NULL,
          field_name      text        NOT NULL,
          extracted_value text,
          corrected_value text,
          status          text        NOT NULL DEFAULT 'approved' CHECK (status IN ('approved', 'rejected')),
          reviewer        text        NOT NULL,
          reviewed_at     timestamptz NOT NULL DEFAULT now(),
          PRIMARY KEY (lease_id, field_name)
        )""")
    conn.execute("COMMENT ON TABLE app.abstract_review IS 'Analyst corrections to extracted lease fields (written by the Granite Peak app)'")
    print("app.abstract_review ready:", conn.execute("SELECT count(*) FROM app.abstract_review").fetchone()[0], "rows")

app.abstract_review ready: 0 rows


In [7]:
# Operational read pattern the app uses: point lookups by key, timed
with pg_connect() as conn:
    ids = [r[0] for r in conn.execute(f'SELECT property_id FROM "{schema}".property_360_synced ORDER BY property_id LIMIT 20')]
    t0 = time.perf_counter()
    for pid in ids:
        row = conn.execute(f'SELECT property_name, msa, occupancy_rate, abr, walt_years FROM "{schema}".property_360_synced WHERE property_id = %s', (pid,)).fetchone()
    ms = (time.perf_counter() - t0) * 1000 / len(ids)
    print(f"property_360 point lookup: {ms:.1f} ms average over {len(ids)} lookups (includes network round trip)")
    print("sample:", row)
    queue = conn.execute(f'''SELECT lease_id, tenant_name, review_status, abstract_confidence
                             FROM "{schema}".lease_abstract_synced
                             ORDER BY abstract_confidence, lease_id LIMIT 5''').fetchall()
    print("review queue head:"); [print("  ", r) for r in queue]

property_360 point lookup: 2.6 ms average over 20 lookups (includes network round trip)
sample: ('Granite Denver Industrial 20', 'Denver', 1.0, 5284145.35, 4.372383416605685)
review queue head:
   ('GP-LSE-00002', 'Pine Legal Group', 'extracted', Decimal('1.00'))
   ('GP-LSE-00009', 'Stone Legal Group', 'extracted', Decimal('1.00'))
   ('GP-LSE-00017', 'Valley Coffee Co', 'extracted', Decimal('1.00'))
   ('GP-LSE-00018', 'Ridge Fitness', 'extracted', Decimal('1.00'))
   ('GP-LSE-00023', 'Summit Materials', 'extracted', Decimal('1.00'))


In [8]:
# Copy analyst corrections from Lakebase to Delta (picked up by the next pipeline update)
import pandas as pd
with pg_connect() as conn:
    rows = conn.execute("SELECT lease_id, field_name, extracted_value, corrected_value, status, reviewer, reviewed_at FROM app.abstract_review").fetchall()
cols = ["lease_id", "field_name", "extracted_value", "corrected_value", "status", "reviewer", "reviewed_at"]
if rows:
    spark.createDataFrame(pd.DataFrame(rows, columns=cols)).createOrReplaceTempView("pg_reviews")
    spark.sql(f"""
      MERGE INTO {uc}.abstract_review t USING pg_reviews s
      ON t.lease_id = s.lease_id AND t.field_name = s.field_name
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *""")
print(f"corrections in Lakebase: {len(rows)}  |  in Delta: {spark.table(f'{uc}.abstract_review').count()}")

corrections in Lakebase: 0  |  in Delta: 0
